# Covid 19 detection from chest X ray images

This notebook aims to distinguish Covid-19, Viral Pneumonia, and normal condition in the chest X ray image dataset using Convolutional Neural Network (CNN) with PyTorch.

[Dataset used](https://www.kaggle.com/datasets/pranavraikokte/covid19-image-dataset)


## 1. Downloading and unzipping the dataset

In [1]:
! export KAGGLE_API_TOKEN=xxxx #Please enter your Kaggle API Token: https://github.com/Kaggle/kaggle-cli/tree/main/docs

In [2]:
#!/bin/bash
! kaggle datasets download pranavraikokte/covid19-image-dataset
! unzip /content/covid19-image-dataset.zip



Dataset URL: https://www.kaggle.com/datasets/pranavraikokte/covid19-image-dataset
License(s): CC-BY-SA-4.0
100% 158M/158M [00:01<00:00, 134MB/s]

Archive:  /content/covid19-image-dataset.zip
  inflating: Covid19-dataset/test/Covid/0100.jpeg  
  inflating: Covid19-dataset/test/Covid/0102.jpeg  
  inflating: Covid19-dataset/test/Covid/0105.png  
  inflating: Covid19-dataset/test/Covid/0106.jpeg  
  inflating: Covid19-dataset/test/Covid/0108.jpeg  
  inflating: Covid19-dataset/test/Covid/0111.jpg  
  inflating: Covid19-dataset/test/Covid/0112.jpg  
  inflating: Covid19-dataset/test/Covid/0113.jpg  
  inflating: Covid19-dataset/test/Covid/0115.jpeg  
  inflating: Covid19-dataset/test/Covid/0118.jpeg  
  inflating: Covid19-dataset/test/Covid/0119.jpeg  
  inflating: Covid19-dataset/test/Covid/0120.jpg  
  inflating: Covid19-dataset/test/Covid/094.png  
  inflating: Covid19-dataset/test/Covid/096.png  
  inflating: Covid19-dataset/test/Covid/098.jpeg  
  inflating: Covid19-dataset/test/Covid

##2. Importing necessary libraries

In [3]:
#Reference: https://www.youtube.com/watch?v=ZBfpkepdZlw
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import os
from scipy import misc
import glob
from PIL import Image
from torchvision import datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.optim as optim
import torch.nn.functional as F

/tmp/ipykernel_12217/1703849811.py:9: DeprecationWarning: scipy.misc is deprecated and will be removed in 2.0.0
  from scipy import misc


##3. Setting the device

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

##4. Preparing dataloaders

In [5]:
#loading data
train_data_dir="/content/Covid19-dataset/train"

test_data_dir="/content/Covid19-dataset/test"
#Reference: https://stackoverflow.com/questions/10439104/reading-bmp-files-in-python



In [6]:
img_size=32
#Reference: https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
train_transform = transforms.Compose([
    transforms.Resize((img_size,img_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

tv_transform = transforms.Compose([
    transforms.Resize((img_size,img_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
#Reference: https://stackoverflow.com/questions/69199273/torchvision-imagefolder-could-not-find-any-class-folder

train_data=datasets.ImageFolder(train_data_dir, transform=train_transform)
test_data=datasets.ImageFolder(test_data_dir,transform=tv_transform)

#128 arbitrary
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)




##5. Constructing the neural network

In [7]:
#Reference: https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        #Reference: https://www.youtube.com/watch?v=CtzfbUwrYGI
        self.conv1 = nn.Conv2d(3, 6, 5) #for the case of 256x256 #(256-5)/1 +1=252 (6,252,252)
        self.pool = nn.MaxPool2d(2, 2) #(6,126,126)
        self.conv2 = nn.Conv2d(6, 16, 5) #(126-5)/1+1 =122 (16,122,122) ->(16,61,61) ->Flatten(16*61*61)
        self.fc1 = nn.Linear(16 * 5* 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()
net.to(device)

Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=3, bias=True)
)

##6. Setting the loss function and optimizer

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01, momentum = 0.9)


##7. Training the model

In [9]:
for epoch in range(30):
  print(f"Training epoch {epoch}...")

  running_loss =0.0
   #Reference: https://zenn.dev/zzz/articles/970e5df5cf1f65
  for i, data in enumerate(train_loader, 0):
    inputs, labels = data[0].to(device), data[1].to(device)

    optimizer.zero_grad()

    outputs = net(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()

  print((f'Loss:{running_loss / len(train_loader):.4f}'))

print("finished Training")


Training epoch 0...
Loss:1.1097
Training epoch 1...
Loss:1.0943
Training epoch 2...
Loss:1.0762
Training epoch 3...
Loss:1.0650
Training epoch 4...
Loss:1.0535
Training epoch 5...
Loss:1.0330
Training epoch 6...
Loss:0.9850
Training epoch 7...
Loss:0.8629
Training epoch 8...
Loss:0.6327
Training epoch 9...
Loss:0.4862
Training epoch 10...
Loss:0.4452
Training epoch 11...
Loss:0.3989
Training epoch 12...
Loss:0.3993
Training epoch 13...
Loss:0.5165
Training epoch 14...
Loss:0.3896
Training epoch 15...
Loss:0.3499
Training epoch 16...
Loss:0.3016
Training epoch 17...
Loss:0.2301
Training epoch 18...
Loss:0.1997
Training epoch 19...
Loss:0.1790
Training epoch 20...
Loss:0.2121
Training epoch 21...
Loss:0.2471
Training epoch 22...
Loss:0.2125
Training epoch 23...
Loss:0.1951
Training epoch 24...
Loss:0.1918
Training epoch 25...
Loss:0.1899
Training epoch 26...
Loss:0.1199
Training epoch 27...
Loss:0.1460
Training epoch 28...
Loss:0.1250
Training epoch 29...
Loss:0.1014
finished Training


##8. Testing the model

In [10]:
torch.save(net.state_dict(), 'trained_net.pth')

In [11]:
net = Net()
net.to(device)
net.load_state_dict(torch.load("trained_net.pth"))

<All keys matched successfully>

In [12]:
#Reference: https://www.youtube.com/watch?v=CtzfbUwrYGI
correct = 0
total = 0
net.eval()

with torch.no_grad():
  for data in test_loader:
    images,labels = data[0].to(device), data[1].to(device)
    outputs=net(images)
    _, predicted = torch.max(outputs,1)
    total+=labels.size(0)
    correct+=(predicted == labels).sum().item()

accuracy = round(100*correct/total,2)

print(f"Accuracy of the model: {accuracy}%")


Accuracy of the model: 84.85%
